# ✅ Feature Scaling — Answer Notebook
## Standardization vs Normalization
**Amol Jagtap | amoljagtap3001@gmail.com**

---

> ⚠️ **Attempt the Practice Notebook FIRST.**
> These are reference solutions — your code may look different and still be correct.


---
## ⚙️ Setup

In [ ]:
# Amol Jagtap | amoljagtap3001@gmail.com
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Ready!')


---
## Section 1: Dataset — ANSWERS

In [ ]:
np.random.seed(42); n=400
df = pd.DataFrame({
    'Age'          : np.random.randint(18,70,n),
    'Income'       : np.random.exponential(scale=35000,size=n)+15000,
    'Credit_Score' : np.random.randint(300,850,n),
    'Years_Exp'    : np.random.randint(0,40,n),
    'Loan_Approved': None
})
df['Loan_Approved'] = np.where(
    (df['Credit_Score']>600)&(df['Income']>30000),1,0)
df.loc[5,'Income']=950000
df.loc[50,'Income']=880000
print(df.head())
print(df.describe())


### 1.2 — Inspect Scale Differences

In [ ]:
numeric_cols = ['Age','Income','Credit_Score','Years_Exp']

for col in numeric_cols:
    print(f'{col:<14}: min={df[col].min():>10.1f}  max={df[col].max():>10.1f}  '
          f'range={df[col].max()-df[col].min():>10.1f}')

# Income has by far the largest range; Age has the smallest


---
## Section 2: Why Scaling Matters — ANSWERS

### 2.1 — Unscaled Boxplot

In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=df[['Age','Income','Credit_Score']])
plt.title('Unscaled Features — Notice the Scale Difference')
plt.show()
# Income's box completely dwarfs Age and Credit_Score visually


### 2.2 — Euclidean Distance Without Scaling

In [ ]:
row0 = df.iloc[0]; row1 = df.iloc[1]
dist = np.sqrt((row0['Age']-row1['Age'])**2 + (row0['Income']-row1['Income'])**2)
print('Distance (Age + Income):', dist.round(2))

age_contribution    = (row0['Age']-row1['Age'])**2
income_contribution = (row0['Income']-row1['Income'])**2
print('Age contribution to distance^2   :', age_contribution)
print('Income contribution to distance^2:', income_contribution)
# Income contributes orders of magnitude more, it completely dominates
# the distance calculation, making Age effectively meaningless to KNN/SVM.


---
## Section 3: Min-Max Normalization — ANSWERS

### 3.1 — Manual Min-Max

In [ ]:
age_min = df['Age'].min()
age_max = df['Age'].max()
df['Age_minmax_manual'] = (df['Age'] - age_min) / (age_max - age_min)
print('New min:', df['Age_minmax_manual'].min())
print('New max:', df['Age_minmax_manual'].max())


### 3.2 — sklearn MinMaxScaler

In [ ]:
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(df[numeric_cols])
print('Learned min:', scaler_mm.data_min_)
print('Learned max:', scaler_mm.data_max_)
print(pd.DataFrame(X_mm, columns=numeric_cols).head())

print('\nMatch check (Age):')
print(pd.DataFrame(X_mm, columns=numeric_cols)['Age'].head().values)
print(df['Age_minmax_manual'].head().values)


### 3.3 — Before/After Visualisation

In [ ]:
income_scaled_mm = MinMaxScaler().fit_transform(df[['Income']])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df['Income'].hist(ax=axes[0], bins=30, color='salmon', edgecolor='white')
axes[0].set_title('Original Income')
pd.Series(income_scaled_mm.flatten()).hist(ax=axes[1], bins=30,
                                            color='steelblue', edgecolor='white')
axes[1].set_title('Min-Max Scaled Income')
plt.tight_layout(); plt.show()
# The SHAPE of the distribution does NOT change, only the scale of the x-axis.
# It is still right-skewed after scaling.


---
## Section 4: Standardization — ANSWERS

### 4.1 — Manual Z-Score

In [ ]:
cs_mean = df['Credit_Score'].mean()
cs_std  = df['Credit_Score'].std()
df['Credit_Score_z_manual'] = (df['Credit_Score'] - cs_mean) / cs_std
print('New mean:', df['Credit_Score_z_manual'].mean().round(4))
print('New std :', df['Credit_Score_z_manual'].std().round(4))


### 4.2 — sklearn StandardScaler

In [ ]:
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(df[numeric_cols])
print('Learned mean :', scaler_std.mean_)
print('Learned scale:', scaler_std.scale_)
print(pd.DataFrame(X_std, columns=numeric_cols).head())

print('\nMatch check (Credit_Score):')
print(pd.DataFrame(X_std, columns=numeric_cols)['Credit_Score'].head().values)
print(df['Credit_Score_z_manual'].head().values)


### 4.3 — Effect on Skewed Feature with Outliers

In [ ]:
income_scaled = StandardScaler().fit_transform(df[['Income']])
print('Min:', income_scaled.min().round(2))
print('Max:', income_scaled.max().round(2))
# The outliers are still extreme (large positive z-scores, e.g. greater than 5)
# because StandardScaler uses mean/std, which outliers heavily influence,
# but it does NOT clip or bound the output, outliers remain outliers.


---
## Section 5: Comparison — ANSWERS

### 5.1 — Comparison Table

In [ ]:
income_orig = df[['Income']].values
income_mm   = MinMaxScaler().fit_transform(income_orig)
income_std  = StandardScaler().fit_transform(income_orig)

comparison = pd.DataFrame({
    'Original': income_orig.flatten(),
    'MinMax'  : income_mm.flatten(),
    'Standard': income_std.flatten()
}).agg(['min','max','mean','std']).round(3)
print(comparison)


### 5.2 — Triple Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(income_orig, bins=30, color='gray', edgecolor='white')
axes[0].set_title('Original')
axes[1].hist(income_mm, bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Min-Max Scaled [0,1]')
axes[2].hist(income_std, bins=30, color='forestgreen', edgecolor='white')
axes[2].set_title('Standardized (mean=0, std=1)')
plt.tight_layout(); plt.show()


---
## Section 6: RobustScaler — ANSWERS

### 6.1 — Compare All 3 Scalers on Outlier Data

In [ ]:
income = df[['Income']].values
scalers = {'MinMax': MinMaxScaler(), 'Standard': StandardScaler(), 'Robust': RobustScaler()}

for name, sc in scalers.items():
    scaled = sc.fit_transform(income)
    print(f'{name:<10} min={scaled.min():>8.3f}  max={scaled.max():>8.3f}  '
          f'row5(outlier)={scaled[5][0]:>8.3f}')

# RobustScaler keeps the bulk of values in a tighter, more usable range
# because median/IQR are not pulled by the outlier the way mean/std/min/max are.


### 6.2 — RobustScaler Formula Check

In [ ]:
median = df['Income'].median()
Q1 = df['Income'].quantile(0.25)
Q3 = df['Income'].quantile(0.75)
IQR = Q3 - Q1

df['Income_robust_manual'] = (df['Income'] - median) / IQR

income_robust_sklearn = RobustScaler().fit_transform(df[['Income']])
print('Manual :', df['Income_robust_manual'].head().values.round(4))
print('sklearn:', income_robust_sklearn[:5].flatten().round(4))
# Values match!


---
## Section 7: MaxAbsScaler — ANSWERS

In [ ]:
from sklearn.preprocessing import MaxAbsScaler

mas = MaxAbsScaler()
years_scaled = mas.fit_transform(df[['Years_Exp']])
print('Min:', years_scaled.min().round(3))
print('Max:', years_scaled.max().round(3))

max_abs = df['Years_Exp'].abs().max()
manual_scaled = df['Years_Exp'] / max_abs
print(manual_scaled.head().values.round(3))
print(years_scaled[:5].flatten().round(3))
# Match confirmed


---
## Section 8: Data Leakage — ANSWERS

### 8.1 — Spot the Bug

In [ ]:
# Bug: scaler.fit_transform(X) is called BEFORE train_test_split.
# This means the scaler learns mean/std using BOTH train and test rows,
# so information about the test set 'leaks' into the scaling parameters.
# The model then gets an unrealistic advantage during evaluation.

X = df[numeric_cols]
y = df['Loan_Approved']

# CORRECT VERSION
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled  = scaler.transform(X_test)         # transform test
print('Correct pipeline complete. Shapes:',
      X_train_scaled.shape, X_test_scaled.shape)


### 8.2 — Demonstrate the Leakage Effect

In [ ]:
X = df[numeric_cols].values
y = df['Loan_Approved'].values

# WRONG WAY
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)
Xtr_w,Xte_w,ytr_w,yte_w = train_test_split(X_scaled_wrong,y,test_size=0.2,random_state=42)
knn_wrong = KNeighborsClassifier().fit(Xtr_w, ytr_w)
acc_wrong = knn_wrong.score(Xte_w, yte_w)

# RIGHT WAY
Xtr_r,Xte_r,ytr_r,yte_r = train_test_split(X,y,test_size=0.2,random_state=42)
scaler_right = StandardScaler()
Xtr_r_s = scaler_right.fit_transform(Xtr_r)
Xte_r_s = scaler_right.transform(Xte_r)
knn_right = KNeighborsClassifier().fit(Xtr_r_s, ytr_r)
acc_right = knn_right.score(Xte_r_s, yte_r)

print(f'Wrong way accuracy: {acc_wrong:.3f}')
print(f'Right way accuracy: {acc_right:.3f}')
# On small/simple datasets the gap may look minor, but the RIGHT way
# is the only version that generalises safely to truly unseen data.


---
## Section 9: Pipeline — ANSWERS

### 9.1 — Build Pipeline

In [ ]:
X = df[numeric_cols]
y = df['Loan_Approved']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])
pipe.fit(X_train, y_train)
print('Test Accuracy:', pipe.score(X_test, y_test).round(3))
print(classification_report(y_test, pipe.predict(X_test)))


### 9.2 — Cross-Validate with Pipeline

In [ ]:
scores = cross_val_score(pipe, X, y, cv=5)
print('CV scores:', scores.round(3))
print(f'Mean: {scores.mean():.3f}  Std: {scores.std():.3f}')
# This is safer because the scaler is re-fit on each fold's training
# portion only, no fold ever sees scaling statistics from its own test data.


---
## Section 10: Mini Project — ANSWERS

In [ ]:
X = df[numeric_cols]
y = df['Loan_Approved']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
models = {
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

results_unscaled = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results_unscaled[name] = model.score(X_test, y_test)

print('Without Scaling:')
for name, acc in results_unscaled.items():
    print(f'  {name:<15}: {acc:.3f}')


In [ ]:
results_scaled = {}
for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_train, y_train)
    results_scaled[name] = pipe.score(X_test, y_test)

print('With Scaling:')
for name, acc in results_scaled.items():
    print(f'  {name:<15}: {acc:.3f}')


In [ ]:
comparison_df = pd.DataFrame({
    'Without Scaling': results_unscaled,
    'With Scaling':    results_scaled
})
print(comparison_df)

comparison_df.plot(kind='bar', figsize=(8, 5), color=['salmon','steelblue'])
plt.title('Effect of Scaling on Different Algorithms')
plt.ylabel('Test Accuracy')
plt.xticks(rotation=0)
plt.legend(title='')
plt.tight_layout()
plt.show()

# Conclusions:
# - KNN and SVM (distance/margin-based) typically benefit the MOST from scaling,
#   because their core computation depends directly on feature magnitude.
# - Decision Tree shows almost NO difference, because it splits on thresholds
#   per feature independently, the scale of the feature does not affect
#   where the best split point is found.
